In [9]:
!pip install git+https://github.com/openai/CLIP.git


Defaulting to user installation because normal site-packages is not writeable
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-e49404se
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-e49404se
  Resolved https://github.com/openai/CLIP.git to commit dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1
  Preparing metadata (setup.py) ... done

[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import torch
import torch.nn.functional as F

def compute_cosine_similarity_batch(FIr, FIt):
    """
    Compute cosine similarity between image-level embeddings and prompt embeddings.
    FIr: [N, D]
    FIt: [N, K, D]
    Returns: [N] - averaged cosine similarity per sample
    """
    FIr_norm = F.normalize(FIr, dim=1)                  # [N, D]
    FIt_norm = F.normalize(FIt, dim=2)                  # [N, K, D]
    
    # Cosine similarity between FIr[i] and each FIt[i, k]
    cosine_sim = torch.einsum('nd,nkd->nk', FIr_norm, FIt_norm)  # [N, K]
    return cosine_sim.mean(dim=1)                       # [N]

def compute_patch_cosine_similarity(FIp, FIt):
    """
    Compute fine-grained similarity between patch-level embeddings and prompt embeddings.
    FIp: [N, P, D]
    FIt: [N, K, D]
    Returns: [N] - mean cosine similarity per sample
    """
    FIp_norm = F.normalize(FIp, dim=2)                  # [N, P, D]
    FIt_norm = F.normalize(FIt, dim=2)                  # [N, K, D]

    # Cosine similarity between each patch and each prompt token
    cosine_sim = torch.einsum('npd,nkd->npk', FIp_norm, FIt_norm)  # [N, P, K]
    return cosine_sim.mean(dim=(1, 2))                  # [N]

def compute_quality_scores(FIr, FIp, FIt, L=5.0):
    """
    Compute final quality scores for each sample in the batch.
    
    Args:
        FIr: [N, D]     - image-level embeddings
        FIp: [N, P, D]  - patch-level embeddings
        FIt: [N, K, D]  - prompt token embeddings
        L: float        - scaling factor (default 5)

    Returns:
        Qfg: [N] - final quality score
        wI: [N]  - image-level fine-grained similarity
        wP: [N]  - patch-level fine-grained similarity
    """
    wI = compute_cosine_similarity_batch(FIr, FIt)  # [N]
    wP = compute_patch_cosine_similarity(FIp, FIt)  # [N]
    Qfg = ((wI + wP) / 2) * L
    return Qfg, wI, wP


In [9]:
# Example shapes
# FIr: [2982, 512]
# FIp: [2982, 196, 512]
# FIt: [2982, 77, 512]
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
FIr = torch.randn(2982, 512).to(device)     # image embeddings
FIp = torch.randn(2982, 196, 512).to(device)  # patch embeddings
FIt = torch.randn(2982, 77, 512).to(device)   # prompt embeddings

Qfg, wI, wP = compute_quality_scores(FIr, FIp, FIt)
print(Qfg.shape)  # should be [2982]
print(wI.shape)   # should be [2982]
print(wP.shape)   # should be [2982]
print(Qfg[:5])  # print quality scores for first few samples

torch.Size([2982])
torch.Size([2982])
torch.Size([2982])
tensor([-0.0089,  0.0111, -0.0069,  0.0162,  0.0124])


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import clip
from torch.optim.lr_scheduler import CosineAnnealingLR
import numpy as np
import os
from uuid import uuid4
from torch.utils.data import DataLoader, random_split
from torchvision.transforms.functional import to_pil_image

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Custom Dataset
class AIGIDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        """
        Args:
            csv_file (str): Path to CSV file with image paths, prompts, and MOS scores.
            img_dir (str): Directory containing images.
            transform: Image transformations.
        """
        self.data = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_name = os.path.join(self.img_dir, self.data.iloc[idx]["name"])
        image = Image.open(img_name).convert("RGB")
        
        # Initial prompt
        initial_prompt = self.data.iloc[idx]["prompt"]
        
        # MOS score (quality score)
        mos_quality = float(self.data.iloc[idx]["mos_quality"])
        
        if self.transform:
            image = self.transform(image)
        
        return {
            "image": image,
            "initial_prompt": initial_prompt,
            "mos_quality": mos_quality
        }

# Image patch extraction
def extract_patches(image, patch_size=64, stride=32, return_tensor=True):
    """
    Extract patches from a [C, H, W] image tensor using sliding window.
    
    Args:
        image (Tensor): Input image of shape [C, H, W].
        patch_size (int): Height and width of each patch.
        stride (int): Stride between patches.
        return_tensor (bool): Whether to return a stacked tensor or list of patches.
    
    Returns:
        Tensor [N, C, patch_size, patch_size] if return_tensor is True,
        else a list of patches.
    """
    patches = []
    # print(image.shape)
    _,c, h, w = image.shape
    for i in range(0, h - patch_size + 1, stride):
        for j in range(0, w - patch_size + 1, stride):
            patch = image[:,:, i:i+patch_size, j:j+patch_size]
            patches.append(patch)
    return torch.stack(patches) if return_tensor else patches


# Task-specific prompt construction
def construct_task_specific_prompts():
    """
    Construct task-specific prompts for alignment and perception quality.
    Returns:
        Dictionary with alignment and perception prompts.
    """
    alignment_adverbs = ["badly", "poorly", "fairly", "well", "perfectly"]
    perception_adjectives = ["bad", "poor", "fair", "good", "perfect"]
    
    alignment_prompts = [f"A photo that {adv} matches {{pt}}." for adv in alignment_adverbs]
    perception_prompts = [f"A photo of {adj} quality." for adj in perception_adjectives]
    
    return {
        "alignment": alignment_prompts,
        "perception": perception_prompts
    }

# TSP-MGS Model
class TSPMGS(nn.Module):
    def __init__(self, clip_model, num_quality_levels=5, alpha=0.5,transform=None):
        super(TSPMGS, self).__init__()
        self.clip_model = clip_model
        self.num_quality_levels = num_quality_levels
        self.alpha = alpha
        self.transform=transform
        
    def forward(self, images, patches_list, task_prompts, initial_prompts,transform):
        """
        Args:
            images (Tensor): Batch of resized images [B, C, H, W].
            patches_list (List): List of patch tensors for each image.
            task_prompts (Dict): Task-specific prompts tokenized.
            initial_prompts (List): Initial prompts for images.
        Returns:
            Predicted quality scores.
        """
        # Extract image and patch features
        image_features = self.clip_model.encode_image(images).float()
        patch_features = []

        for patches in patches_list:
            # Flatten list (in case it's nested)
            if isinstance(patches[0], list):
                patches = [p for sublist in patches for p in sublist]

            # Resize + Normalize each patch
            preprocessed = [transform(to_pil_image(patch)) for batch in patches for patch in batch]
            patch_batch = torch.stack(preprocessed).to(device)  # [N, 3, 224, 224]

            # Encode with CLIP
            with torch.no_grad():
                patch_feats = self.clip_model.encode_image(patch_batch).float()  # [N, D]

            # Average the patch features
            patch_feats = patch_feats.mean(dim=0)  # [D]
            patch_features.append(patch_feats)

        patch_features = torch.stack(patch_features).float()  # [B, D]

        patch_features = torch.stack(patch_features).float()  # [B, 512]
        print(f"Image features shape: {image_features.shape}")
        print(f"Patch features shape: {patch_features.shape}")
        # Extract text features
        align_prompt_feats = self.clip_model.encode_text(task_prompts["alignment"]).float()
        percep_prompt_feats = self.clip_model.encode_text(task_prompts["perception"]).float()
        
        # Tokenize initial prompts and extract word-level features
        initial_tokens = clip.tokenize(initial_prompts).to(device)
        with torch.no_grad():
            text_outputs = self.clip_model.text_model(initial_tokens)
            word_features = text_outputs[1]  # Word-level embeddings
            word_mask = (initial_tokens != 0).float()
        
        # Coarse-grained similarity
        def compute_coarse_similarity(img_feats, txt_feats):
            img_feats = img_feats / img_feats.norm(dim=-1, keepdim=True)
            txt_feats = txt_feats / txt_feats.norm(dim=-1, keepdim=True)
            similarities = img_feats @ txt_feats.T
            probs = torch.softmax(similarities, dim=-1)
            return probs, similarities
        
        # Alignment coarse-grained
        align_img_probs, align_img_sims = compute_coarse_similarity(image_features, align_prompt_feats)
        align_patch_probs, align_patch_sims = compute_coarse_similarity(patch_features, align_prompt_feats)
        
        # Perception coarse-grained
        percep_img_probs, percep_img_sims = compute_coarse_similarity(image_features, percep_prompt_feats)
        percep_patch_probs, percep_patch_sims = compute_coarse_similarity(patch_features, percep_prompt_feats)
        
        # Fine-grained similarity
        def compute_fine_similarity(img_feats, word_feats, word_mask):
            img_feats = img_feats / img_feats.norm(dim=-1, keepdim=True)
            word_feats = word_feats / word_feats.norm(dim=-1, keepdim=True)
            similarities = torch.bmm(img_feats.unsqueeze(1), word_feats.transpose(1, 2)).squeeze(1)
            similarities = similarities * word_mask
            mean_sim = similarities.sum(dim=1) / word_mask.sum(dim=1)
            return mean_sim
        
        img_word_sim = compute_fine_similarity(image_features, word_features, word_mask)
        patch_word_sim = compute_fine_similarity(patch_features, word_features, word_mask)
        
        # Quality regression
        def compute_quality_score(probs, is_patch=False):
            quality_levels = torch.arange(1, self.num_quality_levels + 1, device=device).float()
            score = (self.num_quality_levels / (self.num_quality_levels - 1)) * (
                (probs * quality_levels).sum(dim=-1) - 1
            )
            return score
        
        # Coarse-grained scores
        Q_cg_I_align = compute_quality_score(align_img_probs)
        Q_cg_P_align = compute_quality_score(align_patch_probs, is_patch=True)
        Q_cg_I_percep = compute_quality_score(percep_img_probs)
        Q_cg_P_percep = compute_quality_score(percep_patch_probs, is_patch=True)
        
        # Fine-grained score
        Q_fg = ((img_word_sim + patch_word_sim) / 2) * self.num_quality_levels
        
        # Final quality score
        Q_align = self.alpha * Q_cg_I_align + (1 - self.alpha) * Q_cg_P_align + Q_fg
        Q_percep = self.alpha * Q_cg_I_percep + (1 - self.alpha) * Q_cg_P_percep + Q_fg
        
        # Average alignment and perception for final score (as per AGIQA-1K)
        Q_final = (Q_align + Q_percep) / 2
        
        return Q_final

# Training function
def train_model(model, train_loader, criterion, optimizer, scheduler, num_epochs, task_prompts,transform):
    """
    Train the TSP-MGS model.
    """
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        for batch in train_loader:  
            images = batch["image"].to(device)
            initial_prompts = batch["initial_prompt"]
            mos_qualitys = batch["mos_quality"].to(device).float()
            
            # Extract patches
            patches_list = [extract_patches(img.unsqueeze(0), patch_size=64, stride=32) for img in images]
            patches_list = [[p.to(device) for p in patches] for patches in patches_list]
            
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(images, patches_list, task_prompts, initial_prompts,transform=transform)
            
            # Compute loss
            loss = criterion(outputs, mos_qualitys)
            
            # Backward and optimize
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
        
        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}")
        
        # Step scheduler
        scheduler.step()

# Testing function
def test_model(model, test_loader, task_prompts):
    """
    Test the TSP-MGS model and compute SRCC and PLCC.
    """
    model.eval()
    predictions = []
    ground_truths = []
    
    with torch.no_grad():
        for batch in test_loader:
            images = batch["image"].to(device)
            initial_prompts = batch["initial_prompt"]
            mos_qualitys = batch["mos_quality"].to(device).float()
            
            # Extract patches
            patches_list = [extract_patches(img.unsqueeze(0), patch_size=112, stride=56) for img in images]
            patches_list = [[p.to(device) for p in patches] for patches in patches_list]
            
            # Forward pass
            outputs = model(images, patches_list, task_prompts, initial_prompts)
            
            predictions.append(outputs.cpu().numpy())
            ground_truths.append(mos_qualitys.cpu().numpy())
    
    predictions = np.concatenate(predictions)
    ground_truths = np.concatenate(ground_truths)
    
    # Compute SRCC
    ranks_pred = np.argsort(np.argsort(predictions))
    ranks_gt = np.argsort(np.argsort(ground_truths))
    rank_diff = ranks_pred - ranks_gt
    srcc = 1 - (6 * np.sum(rank_diff ** 2)) / (len(predictions) * (len(predictions) ** 2 - 1))
    
    # Compute PLCC
    pred_mean = np.mean(predictions)
    gt_mean = np.mean(ground_truths)
    num = np.sum((predictions - pred_mean) * (ground_truths - gt_mean))
    den = np.sqrt(np.sum((predictions - pred_mean) ** 2) * np.sum((ground_truths - gt_mean) ** 2))
    plcc = num / den if den != 0 else 0
    
    return srcc, plcc, predictions, ground_truths

# Main function
def main():
    # Hyperparameters
    batch_size = 16
    num_epochs = 20
    learning_rate = 5e-6
    weight_decay = 5e-4
    image_size = 224
    alpha = 0.5  # Balance parameter
    

    transform = transforms.Compose([
        transforms.Resize(224),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.4815, 0.4578, 0.4082],
                            std=[0.2686, 0.2613, 0.2758])
    ])

    
    # Load datasets (replace with your paths)

    # Define your total dataset
    dataset = AIGIDataset(
        csv_file="data.csv",
        img_dir="AGIQA-3K",
        transform=transform
    )

    # Split into train/test (e.g., 80% train, 20% test)
    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size
    train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

    # Create DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

    # Load CLIP model
    clip_model, preprocess = clip.load("ViT-B/32", device=device)
    
    # Initialize TSP-MGS model
    model = TSPMGS(clip_model, num_quality_levels=5, alpha=alpha,transform=transform).to(device)
    
    # Optimizer and scheduler
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=0)
    
    # Loss function
    criterion = nn.L1Loss()  # MAE as per paper
    
    # Construct task-specific prompts
    task_prompt_templates = construct_task_specific_prompts()
    # Tokenize task-specific prompts
    task_prompts = {
        "alignment": clip.tokenize(task_prompt_templates["alignment"]).to(device),
        "perception": clip.tokenize(task_prompt_templates["perception"]).to(device)
    }
    
    # Train model
    print("Starting training...")
    train_model(model, train_loader, criterion, optimizer, scheduler, num_epochs, task_prompts,transform)
    
    # Test model
    print("Starting testing...")
    srcc, plcc, predictions, ground_truths = test_model(model, test_loader, task_prompts,transform)
    print(f"SRCC: {srcc:.4f}, PLCC: {plcc:.4f}")

if __name__ == "__main__":
    main()

Starting training...


TypeError: stack(): argument 'tensors' (position 1) must be tuple of Tensors, not Tensor